### Q1:

In [8]:
import pandas as pd
import re

def load_data(path):
    try:
        df = pd.read_csv(path)
        assert 'review_text' in df.columns
        return df
    except Exception as e:
        print("Error:", e)

df = load_data("shopsense_reviews.csv")
df = df.dropna(subset=['review_text'])

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text.split()

tokenized_reviews = [preprocess(r) for r in df['review_text']]

#### (a):

In [30]:
from gensim.models import Word2Vec

def train_word2vec(sentences, window_size=5):
    model = Word2Vec(
        sentences,
        vector_size=100,
        window=window_size,
        min_count=1,
        workers=4
    )
    return model

w2v_model = train_word2vec(tokenized_reviews)

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(model, w1, w2):
    if w1 not in model.wv or w2 not in model.wv:
        return None
    
    v1 = model.wv[w1].reshape(1, -1)
    v2 = model.wv[w2].reshape(1, -1)
    return cosine_similarity(v1, v2)[0][0]

In [32]:
similar_words = w2v_model.wv.most_similar('cheap', topn=10)

print("Top similar words to 'cheap':")
for word, score in similar_words:
    print(word, ":", score)

Top similar words to 'cheap':
material : 0.9814308881759644
finishing : 0.8588293194770813
price : 0.7839670777320862
worth : 0.7399190068244934
poor : 0.7304753065109253
much : 0.6238659620285034
expected : 0.5827008485794067
bgreat : 0.5822786092758179
productb : 0.5699522495269775
rupee : 0.5122693181037903


In [33]:

affordable_word = None
low_quality_word = None

for word, score in similar_words:
    if word in ['budget', 'value', 'lowcost'] and affordable_word is None:
        affordable_word = word
    if word in ['poor', 'bad', 'weak'] and low_quality_word is None:
        low_quality_word = word

print("Affordable proxy word:", affordable_word)
print("Low-quality proxy word:", low_quality_word)

Affordable proxy word: None
Low-quality proxy word: poor


In [29]:
sim_affordable = cosine_sim(w2v_model, 'cheap', affordable_word)
sim_low_quality = cosine_sim(w2v_model, 'cheap', low_quality_word)

print("cheap vs", affordable_word, ":", sim_affordable)
print("cheap vs", low_quality_word, ":", sim_low_quality)

cheap vs None : None
cheap vs poor : 0.7337226


The word “cheap” is polysemous, meaning it has multiple meanings:

- Affordable (positive context) → low price, good value
- Low-quality (negative context) → poor durability, bad material

Word2Vec assigns only ONE vector per word, regardless of context.

Therefore:

- The vector for "cheap" is a blend of both meanings
- It does not distinguish between:
“cheap and good value”
“cheap and poor quality”

Initially, words like “affordable” and “flimsy” were not found in the vocabulary.

Reason:

- Word2Vec only learns from words present in dataset
- Missing words - Out-of-Vocabulary (OOV) problem

Instead of hardcoding external words, we:

- Used most_similar('cheap')
- Selected dataset-based proxy words:
  - Affordable meaning → budget, value
  - Low-quality meaning → poor, bad

#### (b):

In [35]:
import numpy as np

def sentence_vector(model, sentence):
    words = preprocess(sentence)
    vectors = [model.wv[w] for w in words if w in model.wv]
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

In [37]:
def get_anchor_vector(model, words):
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0)

affordable_anchor = get_anchor_vector(w2v_model, ['budget', 'value', 'lowcost'])
low_quality_anchor = get_anchor_vector(w2v_model, ['poor', 'bad', 'weak'])

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

def classify_cheap(sentence, model):
    sent_vec = sentence_vector(model, sentence).reshape(1, -1)
    
    aff_sim = cosine_similarity(sent_vec, affordable_anchor.reshape(1, -1))[0][0]
    low_sim = cosine_similarity(sent_vec, low_quality_anchor.reshape(1, -1))[0][0]
    
    if aff_sim > low_sim:
        return "Affordable Meaning"
    else:
        return "Low-Quality Meaning"

In [ ]:
s1 = "This phone is cheap and worth the price"
s2 = "The product feels cheap and breaks easily"

print(s1, "-", classify_cheap(s1, w2v_model))
print(s2, "-", classify_cheap(s2, w2v_model))

This phone is cheap and worth the price → Low-Quality Meaning
The product feels cheap and breaks easily → Low-Quality Meaning


The model predicts “Low-Quality Meaning” for both sentences because the anchor words used for the affordable meaning (like “budget” or “value”) are either missing or very rare in the dataset. Due to this, Word2Vec does not learn strong embeddings for them (OOV problem), making the affordable anchor weak. In contrast, low-quality words (like “poor”, “bad”) are more frequent and better learned, so the model shows higher similarity with the low-quality meaning in both cases.

#### (c):

In [40]:
# Train models with different window sizes
model_w2 = train_word2vec(tokenized_reviews, window_size=2)
model_w10 = train_word2vec(tokenized_reviews, window_size=10)

# Compare similar words for 'cheap'
print("Window = 2")
print(model_w2.wv.most_similar('cheap', topn=5))

print("\nWindow = 10")
print(model_w10.wv.most_similar('cheap', topn=5))

Window = 2
[('material', 0.9007120132446289), ('price', 0.8192227482795715), ('worth', 0.7230167388916016), ('bgreat', 0.6560027003288269), ('productb', 0.6519758105278015)]

Window = 10
[('material', 0.9962731003761292), ('finishing', 0.9682300686836243), ('much', 0.9102747440338135), ('price', 0.8222053647041321), ('poor', 0.7926887273788452)]


The window size in Word2Vec defines how many surrounding words are considered as context for a target word.

- Small window (e.g., 2) → looks at nearby words.
    - It captures Syntactic relationships
    - more precise but limited context
- Large window (e.g., 10) → looks at broader context
    - It captures Semantic relationships
    - richer meaning but less focus on exact structure

### Q2:

In [41]:
review_A = "incredible camera but terrible battery life"
review_B = "Battery drains fast, although photos are stunning"

### BOW:

In [42]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def bow_similarity(a, b):
    vec = CountVectorizer()
    X = vec.fit_transform([a, b])
    return cosine_similarity(X[0], X[1])[0][0]

print("BOW Similarity:", bow_similarity(review_A, review_B))

BOW Similarity: 0.1543033499620919


### TF-IDF:

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_similarity(a, b):
    vec = TfidfVectorizer()
    X = vec.fit_transform([a, b])
    return cosine_similarity(X[0], X[1])[0][0]

print("TF-IDF Similarity:", tfidf_similarity(review_A, review_B))

TF-IDF Similarity: 0.0845798608014702


### Word2Vec:

In [44]:
def w2v_similarity(a, b, model):
    v1 = sentence_vector(model, a).reshape(1, -1)
    v2 = sentence_vector(model, b).reshape(1, -1)
    return cosine_similarity(v1, v2)[0][0]

print("Word2Vec Similarity:", w2v_similarity(review_A, review_B, w2v_model))

Word2Vec Similarity: 0.36532936


### Sentence-BERT:

In [45]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def sbert_similarity(a, b):
    emb = sbert_model.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

print("SBERT Similarity:", sbert_similarity(review_A, review_B))

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1398.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SBERT Similarity: 0.6335721


#### (a):

Sentence-BERT gives the highest similarity because it understands the overall meaning of sentences. 
Word2Vec performs moderately well, while TF-IDF and BOW perform poorly.

#### (b):

BOW fails because it relies on exact word matching. The two sentences use different words:

- “camera” vs “photos”
- “battery” vs “drains fast”

Since there is little word overlap, similarity is low despite similar meaning.

#### (c):

The semantic gap is the difference between surface words and actual meaning.

- BOW and TF-IDF cannot capture meaning beyond exact words
- Word2Vec captures word-level meaning
- Sentence-BERT captures full sentence meaning

Thus, each method progressively reduces the semantic gap.